# MeteoScreening `SW_OUT_T1_47_1` (2020-2025) from database (influxdb)

***
**Site**: CH-LAE &nbsp;&nbsp;|&nbsp;&nbsp; **Variable**: `SW_OUT_T1_47_1` &nbsp;&nbsp;|&nbsp;&nbsp; **Sensor**: Kipp &amp; Zonen **CNR4** (SN 212965) from 14 Dec 2021, **CNR1** (SN 020484) before that &nbsp;&nbsp;|&nbsp;&nbsp; **Period**: 2020-2025  
**Derived from**: diive notebook template `DatabaseInfluxStepwiseMeteoScreening.ipynb` (version `10`, 2 Sep 2026)  
**Author**: Lukas Hörtnagl (holukas@ethz.ch)

## ℹ️ About this notebook
Download raw **outgoing (reflected) shortwave radiation** at 47 m from the InfluxDB database, screen and correct it on the **high-resolution** data, resample to 30MIN, and upload the result back to the database. Screening uses `StepwiseMeteoScreeningDb` from [diive](https://github.com/holukas/diive) (`diive/preprocessing/qaqc/meteoscreening.py`); download and upload use diive's in-house InfluxDB engine (`InfluxIO`, in `diive/core/io/db/influx`).

**Flow:** download (`InfluxIO`) → screen on high-res data (`diive`) → correct the nighttime zero offset → resample to 30MIN → upload.

**This is the first screening notebook for `SW_OUT`.** Its coverage is settled: `PLAN.md` §3.2 records that `SW_OUT_T1_47_1` and `LW_OUT_T1_47_1` both begin on **2020-01-02 00:52**, the same timestamp to the minute, and that probes at 2005, 2008, 2010, 2011, 2014, 2016, 2018 and 2019 return nothing for either. The CNR1 outgoing channels were never ingested, so `SW_OUT` — with `ALB` and a four-component `NETRAD` behind it — is a **2020-2025** product. The *Coverage and sensor eras* section re-derives that start date from the database on every run and fails if it moves, rather than restating it here on trust.

**Scope is deliberately narrow.** The raw record is high-resolution and spans six years, so anything done here is expensive and hard to revisit. This notebook removes what is unambiguously wrong at high resolution, removes the pyranometer's nighttime zero offset, and hands over a clean 30MIN series. Everything else is settled later in `30_PRODUCTS/`, **on the half-hourly data**: the albedo cross-check against `SW_IN_T1_47_1`, the CNR1 → CNR4 continuity question, the 24-day window in which the CNR4 was read through logger constants that had not yet been updated, and any gap handling.

**Outlier detection is stepwise:** run a test, inspect its preview plot, then commit it with `mscr.addflag()`. Re-run with different parameters as often as you like before committing. Run only the tests the variable actually needs. At the end all committed flags are aggregated into one overall quality flag `QCF`.

> **Which channel this is.** `SW_OUT_T1_47_1` is the **tower** channel: the downward-facing pyranometer of the four-component radiometer at 47 m, above the canopy, seeing the forest. Measurement `SW` also holds `SW_OUT_BC_M1_2_1`, the subcanopy channel of a second four-component set at the `M1` station, which sees the forest floor. The two are different scenes at different levels and must never be merged.

> **Database access** needs the `influxdb-client` package, which this project pulls in via the **`diive[db]` extra** (declared in `pyproject.toml`). diive *also* ships a `db` dependency group, but dependency groups are local to the project that declares them — `uv sync --group db` only works inside the diive repo, not from here.

## ⏱️ Timestamp convention (important)
The database stores every timestamp in **UTC** and as **`TIMESTAMP_END`** (the stamp marks the *end* of the averaging interval). Getting this right is the one thing that must not go wrong: the value written back to the database depends on it, and so does the day/night split that both the screening and the zero-offset correction rest on. For a radiation variable a timestamp error is not a cosmetic problem — it moves the measured diurnal course against the sun.

The single knob is `TIMEZONE_OFFSET_TO_UTC_HOURS` (set in *User settings*). It is applied **identically** on download and on upload:

| Stage | Timezone | Convention | Done by |
|---|---|---|---|
| Database | UTC | `TIMESTAMP_END` | InfluxDB |
| After `dbc.download(..., timezone_offset_to_utc_hours=N)` | local (UTC+N) | `TIMESTAMP_END` | InfluxIO |
| During screening | local | `TIMESTAMP_MIDDLE` (converted internally) | `StepwiseMeteoScreeningDb` |
| After `mscr.resample()` | local | back to `TIMESTAMP_END` | diive |
| After `dbc.upload_singlevar(..., timezone_offset_to_utc_hours=N)` | UTC | `TIMESTAMP_END` | InfluxIO |

**One place where this bites.** `REMOVE_DATES` is matched against `TIMESTAMP_MID`, not `TIMESTAMP_END`: `StepwiseMeteoScreeningDb` hands the series to `ManualRemoval` *after* the internal conversion, so a record whose `TIMESTAMP_END` is `10:39:00` carries a label half a record earlier inside the test. A bare `'2025-12-19 10:39:00'` would therefore match **nothing** and the removal would silently do nothing. Always give a `[start, stop]` window that brackets the record, never a bare timestamp; it only matters when removing individual records, since a window spanning days has edges far larger than the half-record offset.

The *Analyses* section runs an independent check on all of this: a daily correlation of the measured series against **potential** radiation, which is astronomical and cannot drift.

## ✏️ User settings (please adjust)

Adjust these before running. What each setting means:

**Site**
- `SITE`, `SITE_LAT`, `SITE_LON`: site ID and coordinates. The coordinates set the day/night split used during screening **and** the one the nighttime zero-offset correction uses to decide which records must read zero. They are not decorative here.

**Variable to screen**
- `FIELD`: the InfluxDB `_field`, assembled into the one-element list `FIELDS` that `StepwiseMeteoScreeningDb` expects.
- `MEASUREMENT`: exactly **one** measurement grouping the variables — `SW` for shortwave radiation.

**Time range to screen**
- `START`: first timestamp to screen — **is** included.
- `STOP`: upper bound — **is not** included.

**Data settings**
- `TIMEZONE_OFFSET_TO_UTC_HOURS`: the critical timestamp knob — see *Timestamp convention*. Must match how the raw data was logged (`1` for CET winter time) and must be the same value everywhere.
- `DATA_VERSION`: the source data version in the database (`raw`).
- `DIRCONF`: local folder holding the database connection config.

**Radiometer eras**
- `CNR4_INSTALLED`, `CNR4_CONSTANTS`: the two dates that split this record into instrument eras. Used by the coverage report and by the per-era counts, not by any correction.

**Resampling**
- `RESAMPLING_FREQ` / `RESAMPLING_AGG`: a radiative flux density is a **state** at the half hour, so `'mean'` — never `'sum'`.

**Physical range**
- `SW_OUT_MIN`, `SW_OUT_MAX`: the absolute-limits test. The reasoning behind both numbers is in the settings cell, and the cell before the test checks them against what this record actually contains.

**Parameter help**
- `SHOW_PARAM_HELP`: `True` prints the full docstring of each screening method right before it runs.

In [ ]:
# --- Site ---
SITE = 'ch-lae'
SITE_LAT = 47.478333  # CH-LAE
SITE_LON = 8.364389  # CH-LAE

# --- Variable to screen ---
# (!) This is the TOWER channel at 47 m, above the canopy. Measurement SW also holds
#     SW_OUT_BC_M1_2_1, the subcanopy channel of a second four-component set at the M1
#     station, which sees the forest floor. Different scene, different level, never merge.
FIELD = 'SW_OUT_T1_47_1'
FIELDS = [FIELD]  # StepwiseMeteoScreeningDb expects a list
MEASUREMENT = 'SW'

# --- Time range to screen ---
# The record begins 2020-01-02 00:52 (PLAN.md section 3.2, re-derived below), so asking for
# 2020-01-01 simply starts at the first record there is. STOP reaches one second into 2026
# because the last bin of 31 Dec 2025 carries TIMESTAMP_END 2026-01-01 00:00.
START = '2020-01-01 00:00:01'  # included
STOP = '2026-01-01 00:00:01'  # not included

# --- Data settings ---
DATA_VERSION = 'raw'
TIMEZONE_OFFSET_TO_UTC_HOURS = 1  # UTC+01:00 (CET, winter time). Must match how the raw data was logged.
DIRCONF = r'F:\dev\poet\configs'
# DIRCONF = r'P:\Flux\RDS_calculations\_scripts\_configs\configs'

# --- Radiometer eras (docs/Instrumentation.md) ---
# The CNR4 (SN 212965) replaced the CNR1 (SN 020484) on 14 Dec 2021. The logger program
# received the CNR4 sensitivities only on 7 Jan 2022 - SW_OUT 14.38 uV W-1 m2 - so the
# 24 days between the two dates are unverified: if the new instrument was read through the
# old multiplier, those values are wrong by the ratio of the sensitivities. This notebook
# does NOT remove that window. A calibration question is decided on the half-hourly series
# against the other three components, in 30_PRODUCTS, not by deleting data here.
CNR4_INSTALLED = '2021-12-14'
CNR4_CONSTANTS = '2022-01-07'

# --- Resampling ---
RESAMPLING_FREQ = '30min'  # screened high-res data is resampled to this frequency
RESAMPLING_AGG = 'mean'  # (!) a flux density is a state at the half hour, never summed

# --- Physical range of reflected shortwave at this site, in W m-2 ---
# Upper limit: reflected radiation cannot exceed incoming radiation. SW_IN at this tower
# peaks near 1100 W m-2, and a mixed forest canopy returns roughly 5-15 % of it, rising for
# the few days a year that snow is held in the canopy. 500 therefore sits far above anything
# this scene can produce while still catching decode and wiring artefacts, which land orders
# of magnitude away rather than just above the physical ceiling.
# Lower limit: deliberately NOT the physical floor of zero. An unventilated pyranometer reads
# a few W m-2 negative at night through its own thermal offset. That offset is real instrument
# behaviour and it is the signal the nighttime zero-offset correction is estimated from - a
# minval of 0 here would delete it and leave that correction nothing to work with.
# (!) Verify both against this channel's own extremes on the first run rather than trusting
#     the paragraph above: the cell before the absolute-limits test prints them and asserts
#     that neither limit has been set inside the ordinary range of the record.
SW_OUT_MIN, SW_OUT_MAX = -50, 500

# --- Parameter help ---
SHOW_PARAM_HELP = False

## 🤖 Auto settings

### Buckets (do not adjust)

In [ ]:
BUCKET_RAW = f'{SITE}_raw'  # source bucket, e.g. 'ch-lae_raw'
BUCKET_PROCESSED = f'{SITE}_processed'  # destination bucket, e.g. 'ch-lae_processed'
print(f'Screening variable:             {FIELD}')
print(f'Source bucket (raw data):       {BUCKET_RAW}')
print(f'Destination bucket (processed): {BUCKET_PROCESSED}')

### Imports

In [ ]:
import importlib.metadata
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

import diive as dv
from diive.core.io.db.influx import InfluxIO  # needs influxdb-client, via the diive[db] extra

warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive v{importlib.metadata.version('diive')}")

## ⬇️ Download data from database

### Connect to database

In [ ]:
dbc = InfluxIO(dirconf=DIRCONF)

List all fields available in the measurement. Worth running here rather than leaving commented: `SW` holds five fields, and two of them are outgoing channels at different levels (`SW_OUT_T1_47_1` above the canopy, `SW_OUT_BC_M1_2_1` below it). Confirming which names exist is also the cheapest guard against screening a mistyped field, which would download nothing and look like a variable that has no data.

In [ ]:
display(dbc.show_fields_in_measurement(bucket=BUCKET_RAW, measurement=MEASUREMENT))

### Coverage and sensor eras
**Run this before anything else, and read it.** `show_field_overview()` queries the field's first and last record over the **whole** database, independently of `START` and `STOP`, and returns its tags. Two things are being established:

1. **That the record still begins where `PLAN.md` says it does**, 2020-01-02 00:52. The question of how far back this field reaches was open while the notebook was written and is now settled: the CNR1 outgoing channels were never ingested, so `SW_OUT` is a 2020-2025 product. What remains useful is the guard — a re-ingestion that moved the start would change the product period and every statement resting on it, so it is asserted rather than reported.
2. **What the units tag says**, checked against the magnitude of the data further down. A field name is a statement about the variable, not about the instrument, and this one spans two radiometers.

Timestamps here are **UTC**, as the database stores them — the local-time shift is applied on download, not here.

In [ ]:
_ov = dbc.show_field_overview(bucket=BUCKET_RAW, measurement=MEASUREMENT, field=FIELD,
                              data_version=DATA_VERSION)
print(f"Record of {FIELD} in {BUCKET_RAW}: {_ov['first']} to {_ov['last']} (UTC), "
      f"{_ov['n_series']} series")
for _tag, _vals in _ov['tags'].items():
    print(f'  {_tag}: {_vals}')

# A tag can carry more than one value within one data version (different gain, raw_varname
# or freq per era), so everything reading tags has to handle a list rather than a scalar.
assert _ov['first'] is not None, f'(!) {FIELD} holds no records at all in {BUCKET_RAW}'
_first = pd.Timestamp(_ov['first'])
print()
# Settled in PLAN.md section 3.2 and asserted here rather than re-decided: both outgoing
# channels begin on this timestamp, so the record spans BOTH radiometers and 30_PRODUCTS
# has a continuity question to answer at the December 2021 instrument change.
# (!) UTC, as show_field_overview returns it. PLAN.md quotes this start as
# 2020-01-02 00:52, which is the same instant in local time (UTC+1).
RECORD_STARTS = '2020-01-01 23:52:00'
print(f'The record starts {_first} (UTC), '
      f'{(pd.Timestamp(CNR4_INSTALLED) - _first).days} days before the CNR4 was installed '
      f'({CNR4_INSTALLED}), so it spans BOTH radiometers.')
assert _first == pd.Timestamp(RECORD_STARTS), (
    f'(!) the record now starts {_first}, not {RECORD_STARTS} as PLAN.md section 3.2 states. '
    f'The product period, ALB and a four-component NETRAD all rest on that date - settle it '
    f'before screening.')
print(f'-> PASSED, unchanged since PLAN.md recorded it')

### Download
Returns three objects:
- `data_simple`: high-res time series, one column per variable (nice to look at).
- `data_detailed`: dict `{varname: DataFrame}` with each variable's time series **and its database tags** — this is what the screening consumes.
- `assigned_measurements`: the auto-detected measurement per variable (a sanity check).

In [ ]:
%%time
data_simple, data_detailed, assigned_measurements = dbc.download(
    bucket=BUCKET_RAW,
    measurements=[MEASUREMENT],
    fields=FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version=DATA_VERSION,
)

### Inspect downloaded data

In [ ]:
data_simple

In [ ]:
assigned_measurements

Drop any requested variable that has no data in this period:

In [ ]:
vars_not_available = [v for v in FIELDS if v not in data_detailed.keys()]
for rem in vars_not_available:
    FIELDS.remove(rem)
    print(f'Removed {rem} from FIELDS (no data in this period).')
print(f'Data available for: {list(data_detailed.keys())}')
assert FIELD in data_detailed, f'(!) {FIELD} returned no data - nothing to screen'

### Verify download timestamps
Confirm the timestamps look right: **local time** (UTC+`TIMEZONE_OFFSET_TO_UTC_HOURS`), marking the **end** of each averaging interval. The database itself stores UTC — `InfluxIO` applied the offset on download. Eyeball the first/last stamps against the `START`/`STOP` you requested, and note the detected frequency: every window written below as `60 * 24 * …` assumes it.

In [ ]:
for v in data_detailed.keys():
    idx = data_detailed[v].index
    print(f'{v}: index name={idx.name!r}, tz={idx.tz}, freq={idx.freqstr}')
    print(f'   first={idx[0]}   last={idx[-1]}')
print(f'\nApplied UTC offset: +{TIMEZONE_OFFSET_TO_UTC_HOURS}h (timestamps above are local time)')

### Records per instrument era
How much of the downloaded period each radiometer contributed, and how many records fall in the 24-day window in which the CNR4 was in place but the logger still held the CNR1 constants. A test's flagged count is only interpretable **per era**: two instruments with different sensitivities produce different noise, and a statistic over their union describes neither.

If the unverified window is empty the question does not arise for this variable. If it is not, the records stay in — see the note in the settings cell — and the count below is what `30_PRODUCTS` has to account for.

In [ ]:
_s = data_detailed[FIELD][FIELD]
_cnr4, _consts = pd.Timestamp(CNR4_INSTALLED), pd.Timestamp(CNR4_CONSTANTS)
CNR1_ERA = _s.index < _cnr4
UNVERIFIED_ERA = (_s.index >= _cnr4) & (_s.index < _consts)  # CNR4 in place, CNR1 constants in the logger
CNR4_ERA = _s.index >= _consts

_rows = []
for _name, _mask in [(f'CNR1, before {CNR4_INSTALLED}', CNR1_ERA),
                     (f'CNR4, old logger constants, {CNR4_INSTALLED} to {CNR4_CONSTANTS}', UNVERIFIED_ERA),
                     (f'CNR4, own constants, from {CNR4_CONSTANTS}', CNR4_ERA)]:
    _era = _s[_mask]
    _rows.append({'era': _name,
                  'records': len(_era),
                  'with value': int(_era.count()),
                  'coverage %': round(100 * _era.count() / len(_era), 2) if len(_era) else np.nan,
                  'min': round(_era.min(), 2) if _era.count() else np.nan,
                  'max': round(_era.max(), 2) if _era.count() else np.nan})
display(pd.DataFrame(_rows).set_index('era'))

print(f'Records in the unverified 24-day window: {int(_s[UNVERIFIED_ERA].count()):,} '
      f'(kept - a calibration question is settled on the 30MIN series in 30_PRODUCTS)')

### Plot downloaded high-res data

In [ ]:
for varname, frame in data_detailed.items():
    dv.plotting.TimeSeries(series=frame[varname]).plot()

## ▶️ Start MeteoScreening with `diive`

In [ ]:
mscr = dv.qaqc.StepwiseMeteoScreeningDb(
    site=SITE,
    data_detailed=data_detailed,
    fields=FIELDS,
    site_lat=SITE_LAT,
    site_lon=SITE_LON,
    utc_offset=TIMEZONE_OFFSET_TO_UTC_HOURS,
)
mscr.showplot_orig()

## 🔍 Outlier detection
Run a test → inspect its preview → commit with `mscr.addflag()`. Only the committed flag of the **most recent** test is added. Skip any test a variable does not need.

**What is committed here**: **absolute limits** and the **missing-values** flag, plus **manual removal** if windows are entered. Why the statistical tests are off:

- **They act before the zero-offset correction, on the records that correction needs.** The values a z-score or a Hampel filter flags most readily on a radiation series are the small negative nighttime readings — and those are not errors, they are the pyranometer's thermal offset, which the correction in the next section estimates *from them*. Removing them does not merely lose records: it biases the per-day offset and therefore every corrected daytime value of that day.
- **A distribution-wide test measures the sun angle, not a fault.** A z-score over the whole record asks how far a value sits from the multi-year mean; 0 W m-2 at midnight and 150 W m-2 at midday in June are both correct.
- **Reflected shortwave is a product of two varying quantities**, incoming radiation and canopy reflectance, so its increments are legitimately large and abrupt — a cloud edge moves it in a minute, and so does snow falling out of the canopy. Differencing-based tests have no quiet baseline here to scale a threshold against.
- **The day/night switch is `separate_day_night`.** diive v0.91.0 unified the name across every method; the older `separate_daytime_nighttime` raises an error naming its replacement. The template notebook this one is derived from still carries the old name in a few cells.

If you do enable one of them, report its flagged count **per era** (CNR1, CNR4), never as a total.

In [ ]:
mscr.start_outlier_detection()

Plot the current cleaned data at any point during detection:

In [ ]:
for key, val in mscr.outlier_detection.items():
    val.showplot_cleaned(interactive=False)

### Manual removal
Flag specific timestamps or time ranges for removal — known sensor failures, maintenance windows, logger artefacts. Give `[start, stop]` pairs.

**Empty on the first pass, and that is not the same as "nothing is wrong".** No removal window has been established for this channel yet, because no notebook has ever looked at it. Work through the high-res plot above and the QCF heatmaps below, and where something is removed, write down here *what* it was and *what evidence* dated it — a window is the only thing in this notebook that deletes data on a person's say-so.

Two candidates worth checking specifically, neither of which any automatic test can see:

- **The 14 Dec 2021 to 7 Jan 2022 window**, if the record covers it. A wrong multiplier produces perfectly smooth, in-range, plausible values. It is not removed here — see the settings cell.
- **Anything that survives the absolute limits while tracking the sun in the wrong proportion.** Reflected radiation that is a fixed fraction of incoming is what this sensor is supposed to produce; a period where that fraction steps is an albedo question for `30_PRODUCTS`, not a removal.

> **The windows must bracket the record, never name it.** `ManualRemoval` matches against `TIMESTAMP_MID` (see *Timestamp convention*), so a bare timestamp selects nothing and removes nothing, silently. The guard below rejects bare entries for that reason.

> ⚠️ **`REMOVE_DATES` belongs to a sensor, not to a variable.** These notebooks are copies of each other with a few settings changed. Never carry a window across from the `SW_IN` or `LW_IN` notebook because it was written about "the radiometer": the four channels of a CNR are four detectors, and a fault in one is not evidence about the others. Verify it on the preview plot here.

In [ ]:
if SHOW_PARAM_HELP:
    help(dv.outliers.ManualRemoval)

In [ ]:
REMOVE_DATES = [
    # ['2021-12-14 00:00:00', '2021-12-15 00:00:00'],  # window, with the reason for it here
]

# A bare timestamp brackets no middle stamp and removes nothing at all, silently - so
# check the shape before the test runs rather than reading a flagged count of zero later.
for _w in REMOVE_DATES:
    assert isinstance(_w, (list, tuple)) and len(_w) == 2, (
        f'(!) {_w} is not a [start, stop] window - a bare timestamp matches no TIMESTAMP_MID')
    assert pd.Timestamp(_w[1]) > pd.Timestamp(_w[0]), f'(!) window {_w} does not run forwards'

if REMOVE_DATES:
    mscr.flag_manualremoval_test(remove_dates=REMOVE_DATES, showplot=True, verbose=True)
else:
    print('No manual removal windows for this sensor - skip the addflag cell below.')

In [ ]:
if REMOVE_DATES:
    mscr.addflag()

### Absolute limits
Flags values outside the fixed physical range `[SW_OUT_MIN, SW_OUT_MAX]`. This is the workhorse test for this channel: an artefact in a radiation record is either a decode or wiring failure, which lands far outside anything the scene can produce, or it is a calibration question, which absolute limits cannot see and must not be asked to.

The cell below checks the two limits against the record before committing them. A limit set inside the ordinary range of the data is mis-set, not strict — it would fire on the record simply being itself.

> ⚠️ **Do not clip instead.** `correction_setto_max_threshold(threshold=500)` would turn a decode artefact into a fabricated 500 W m-2 — a value inside the physical range that nothing downstream could identify as fake. A removed value is honest; a clipped one is not. The same argument applies at the bottom end and is why the nighttime negatives are *corrected* by an estimated offset rather than clamped where they sit.

In [ ]:
if SHOW_PARAM_HELP:
    help(dv.outliers.AbsoluteLimits)

In [ ]:
# What this record actually contains, before the limits are applied to it.
_orig = mscr.series_hires_orig[FIELD]
print(f'{FIELD}: {_orig.count():,} records with a value, '
      f'range {_orig.min():.2f} to {_orig.max():.2f} W m-2')
display(_orig.describe(percentiles=[.001, .01, .5, .99, .999]).round(2))
_below = int((_orig < SW_OUT_MIN).sum())
_above = int((_orig > SW_OUT_MAX).sum())
print(f'Outside [{SW_OUT_MIN}, {SW_OUT_MAX}]: {_below:,} below, {_above:,} above '
      f'({(_below + _above) / _orig.count():.4%} of records with a value)')
print(f'Negative records: {int((_orig < 0).sum()):,} '
      f'(expected - the nighttime thermal offset, corrected later, not removed here)')

# A threshold inside the record's own ordinary range is mis-set rather than strict. These
# two guards are meant to be able to fire: if they do, look at the extremes above and decide
# whether the limit or the assumption behind it is wrong - do not simply widen the limit.
assert _orig.quantile(.999) < SW_OUT_MAX, (
    f'(!) SW_OUT_MAX={SW_OUT_MAX} sits below the 99.9th percentile of the record')
assert _orig.quantile(.001) > SW_OUT_MIN, (
    f'(!) SW_OUT_MIN={SW_OUT_MIN} sits above the 0.1th percentile of the record')

In [ ]:
mscr.flag_outliers_abslim_test(minval=SW_OUT_MIN, maxval=SW_OUT_MAX, showplot=True, verbose=True)

In [ ]:
mscr.addflag()

### Other tests (all off)
Left switched off for the reasons given at the top of *Outlier detection*. Parameters below are a starting point, not a recommendation: read that section before enabling any of them, run each one against its preview plot, and check what it does to the **nighttime negatives** in particular — those are the input to the correction in the next section.

Note also that differencing-based tests compute their differences **after dropping missing records**, so the records flanking every gap are compared across the gap and look like spikes.

In [ ]:
# Optional, all off by default - read the note above before enabling any of these.
# mscr.flag_outliers_hampel_test(window_length=60 * 24, n_sigma=8, use_differencing=False,
#                                separate_day_night=True, n_sigma_daytime=8, n_sigma_nighttime=12,
#                                repeat=False, showplot=True, verbose=True)
# mscr.flag_outliers_zscore_test(thres_zscore=4.5, separate_day_night=True,
#                                repeat=True, showplot=True, verbose=True)
# mscr.flag_outliers_zscore_rolling_test(thres_zscore=4.5, winsize=60 * 24 * 7,
#                                        repeat=True, showplot=True, verbose=True)
# mscr.flag_outliers_localsd_test(n_sd=7, winsize=60 * 24 * 7, constant_sd=False,
#                                 separate_day_night=True, repeat=False, showplot=True, verbose=True)
# mscr.flag_outliers_increments_zcore_test(thres_zscore=40, repeat=True, showplot=True, verbose=True)
# mscr.addflag()

### Missing values
Not an outlier test — flags missing records so they are counted in the overall `QCF`.

In [ ]:
mscr.flag_missingvals_test(verbose=True)

### Overall quality flag QCF
Aggregate all committed test flags into one overall flag `QCF` (0 = good, 1 = marginal, 2 = bad) and filter the series. Required before corrections and resampling.

In [ ]:
mscr.finalize_outlier_detection()

#### Reports

In [ ]:
mscr.report_outlier_detection_qcf_evolution()

In [ ]:
mscr.report_outlier_detection_qcf_flags()

In [ ]:
mscr.report_outlier_detection_qcf_series()

#### Plots

In [ ]:
mscr.showplot_outlier_detection_qcf_heatmaps()
# mscr.showplot_outlier_detection_qcf_timeseries()

## 📈 Analyses

**This section runs before *Corrections*, which is the reverse of the template order, and the reason is the diagnostic itself.** The nighttime zero-offset correction sets every nighttime record to exactly zero. Potential radiation is also exactly zero at night, so after the correction every night contributes a perfectly matched constant to the daily correlation and every day's correlation rises. Run against the corrected series the test would be partly measuring its own correction; run here, against the QCF-filtered but otherwise untouched series, it measures the sensor.

### Check for timestamp shifts against potential radiation
Compares the measured series to **potential** (clear-sky) radiation, which is astronomical and cannot drift, and returns the correlation for each day. A consistent phase offset between the two is a timestamp error — the one failure mode that no value-based test can detect and that would silently propagate into every flux calculation using this variable.

**Read it differently than for `SW_IN`.** Reflected radiation is incoming radiation times a reflectance that varies with canopy wetness, snow and sun angle, so daily correlations here are lower and more scattered than for an incoming channel even when nothing is wrong. What matters is not the absolute level but its **stability**: an isolated block of low-correlation days, or a step in the daily correlation at a date, is the signal. Individual low days are usually weather.

In [ ]:
%%time
_daycorrs = mscr.analysis_potential_radiation_correlation(
    utc_offset=TIMEZONE_OFFSET_TO_UTC_HOURS, mincorr=0.7, showplot=True)

In [ ]:
# Median daily correlation per year: a step here is a timestamp or mounting question,
# a low overall level is the nature of a reflected channel. Judge the stability, not the level.
_c = _daycorrs[FIELD].dropna()
display(_c.groupby(_c.index.year).agg(['count', 'median', 'min']).round(3))

## 🔧 Corrections
Applied to the high-res, QCF-filtered data. **One correction applies to this variable**, the nighttime zero offset, and it is run below rather than left commented: it is not optional tidying, it is what makes the series physically interpretable. Every other correction diive offers writes a value this sensor did not measure and stays off.

**Order matters and is not arbitrary:** the value inspection comes *before* the correction, and the missing-record mask is captured *before* it too. Both would be meaningless afterwards, for reasons given at each cell.

In [ ]:
mscr.showplot_cleaned()

### Inspect the most frequent values — before the correction
Read-only and safe to run. **It has to run before the zero-offset correction**, which sets every nighttime record to exactly `0`: afterwards `0` is the single most frequent value by construction, and the inspection can no longer tell a stuck channel from the correction's own output.

What would be a fault here is a single value repeating far more often than its neighbours in the distribution — a stuck output. A cluster of small values around zero is not: that is night, seen before it has been zeroed.

In [ ]:
for ff in mscr.fields:
    vc = mscr.series_hires_cleaned[ff].value_counts()
    print(f'--- {ff} (top 20 of {mscr.series_hires_cleaned[ff].count():,} records, '
          f'{mscr.series_hires_cleaned[ff].nunique():,} distinct) ---')
    print(vc.head(20))

### Remove the nighttime zero offset
A pyranometer reads a small non-zero value in the dark: the detector exchanges longwave radiation with the sky and its own body, and an unventilated CNR does this by several W m-2, varying with conditions. `correction_remove_nighttime_zero_offset()` estimates that offset **per day** as the mean of that day's nighttime records (days without nighttime data fall back to the median daily offset), subtracts it from all of that day's records, then sets nighttime to exactly zero and clamps any remaining negative to zero.

Two consequences of the implementation that must be handled here rather than discovered later:

1. **It writes values into records that were never measured.** Setting nighttime to zero is applied to the whole nighttime index, missing records included, so every nighttime gap — including the ones this notebook's own screening just created, and the ones where the station was down — comes back as an exact `0` that looks measured. The mask below is therefore captured **before** the correction and re-applied **after** it, so the uploaded series contains only records that exist. Zero-filling the nights is a defensible product decision; it is `30_PRODUCTS`' decision, taken on the half-hourly series where it can be flagged, not a side effect swallowed here. This is the same trap that put 7,543 modelled values into `30_PRODUCTS/03` looking measured.
2. **The offset is a drift, not a constant.** The per-year table below is the evidence for that and is worth keeping in the record: a nighttime offset that changes across the CNR1 → CNR4 boundary is instrument identity showing up in the data.

The guards after the correction state what it did: nighttime exactly zero, no negatives left, no value written where nothing was measured.

In [ ]:
# The day/night split is defined exactly as the correction defines it internally
# (DaytimeNighttimeFlag on potential radiation), so the checks below cannot disagree
# with the correction about which records are night.
_dnf = dv.variables.DaytimeNighttimeFlag(
    timestamp_index=mscr.series_hires_cleaned[FIELD].index,
    nighttime_threshold=0.001, lat=SITE_LAT, lon=SITE_LON,
    utc_offset=TIMEZONE_OFFSET_TO_UTC_HOURS)
NIGHT = _dnf.get_nighttime_flag() == 1

# (!) Captured BEFORE the correction: it writes an exact zero into every nighttime record,
#     measured or not. Re-applied immediately after, so nothing is uploaded that was not measured.
WAS_MISSING = mscr.series_hires_cleaned[FIELD].isna()

_before = mscr.series_hires_cleaned[FIELD].copy()
_nt = _before[NIGHT].dropna()
print(f'Nighttime records with a value: {len(_nt):,} of {int(NIGHT.sum()):,}')
print('Nighttime offset per year, W m-2 (this is what the correction removes):')
display(_nt.groupby(_nt.index.year).agg(['count', 'median', 'mean', 'min', 'max']).round(2))

In [ ]:
mscr.correction_remove_nighttime_zero_offset()

In [ ]:
# Restore the gaps the correction filled with zeros. series_hires_cleaned is the dict the
# class itself reads at resample(), so assigning into it is how the correction is amended.
_after = mscr.series_hires_cleaned[FIELD]
assert _after.index.equals(_before.index), '(!) the correction returned a different index'
_written_into_gaps = int((WAS_MISSING & _after.notna()).sum())
_after = _after.mask(WAS_MISSING)
mscr.series_hires_cleaned[FIELD] = _after
print(f'Restored {_written_into_gaps:,} records the correction had filled with an exact zero '
      f'({_written_into_gaps / len(_after):.2%} of the period), all of them nighttime gaps.')

# What the correction did, asserted rather than assumed.
_after_nt = _after[NIGHT].dropna()
assert (_after_nt == 0).all(), '(!) nighttime records are not all exactly zero after the correction'
assert (_after.dropna() >= 0).all(), '(!) negative values survived the correction'
assert _after.notna().sum() == _before.notna().sum(), (
    '(!) the number of measured records changed - the gap mask was not restored correctly')
_day = ~NIGHT
print(f'Daytime mean {_before[_day].mean():.2f} -> {_after[_day].mean():.2f} W m-2 '
      f'({_after[_day].mean() - _before[_day].mean():+.2f})')
print(f'Records with a value: {_before.count():,} -> {_after.count():,} (unchanged, as asserted)')

In [ ]:
mscr.showplot_cleaned()

### Corrections that do not apply
All commented out on purpose. Each of them writes a value this sensor did not measure, and none of them has a reason to run on reflected shortwave.

> ⚠️ **Do not set the exact value 0 to missing.** After the correction above, `0` is what every nighttime record legitimately holds. `correction_set_exact_value_to_missing(values=[0])` would delete the entire night of the record.

In [ ]:
# All commented out on purpose - none of these apply to reflected shortwave.
# mscr.correction_remove_relativehumidity_offset()   # RH only
# mscr.correction_setto_max_threshold(threshold=500)  # would fabricate values, see Absolute limits
# mscr.correction_setto_min_threshold(threshold=0)   # would fabricate values, see Absolute limits
# mscr.correction_setto_value(dates=[['2022-04-01', '2022-04-05']], value=0, verbose=1)
# mscr.correction_set_exact_value_to_missing(values=[0])  # (!) would delete every night

## 🔁 Resampling

### Resample to 30MIN
Resample the screened high-res series to `RESAMPLING_FREQ`. The output timestamp is `TIMESTAMP_END` again (see *Timestamp convention*), ready for upload. **This is the handover point**: everything downstream of here — the albedo cross-check against `SW_IN`, the instrument-continuity question, any gap handling — works on the half-hourly series.

In [ ]:
# mincounts_perc stays at the template default of .25: a 30MIN mean of a radiative flux
# density is well estimated from a quarter of its records. It is not a free choice in the
# other direction - a *sum* from a quarter of the records would be a quarter of the truth,
# which is why the precipitation notebook chose differently.
mscr.resample(to_freqstr=RESAMPLING_FREQ, agg=RESAMPLING_AGG, mincounts_perc=.25)
mscr.showplot_resampled()

### Check the resampled time resolution

In [ ]:
# This runs before the upload, so a wrong resolution must stop the notebook rather than
# print a warning that scrolls past and is then written to the database anyway.
for v in mscr.resampled_detailed.keys():
    freq = dv.times.DetectFrequency(index=mscr.resampled_detailed[v].index, verbose=True).get()
    print(f'{v}: {freq}')
    assert freq == RESAMPLING_FREQ, (
        f'(!) {v} resampled to {freq}, not {RESAMPLING_FREQ} - do not upload this')
print(f'-> PASSED, every series is {RESAMPLING_FREQ}')

### Check the resampled series before it is uploaded
The last chance to catch a screening that removed something real. Three things are asserted: the series is still non-negative, no half hour exceeds the physical ceiling, and the nights are still zero. The per-year table beside them is for reading, not for asserting — a year that steps against its neighbours is a question for `30_PRODUCTS`, not a reason to stop the upload.

In [ ]:
_res = mscr.resampled_detailed[FIELD][FIELD]
assert (_res.dropna() >= 0).all(), '(!) negative half-hourly values - the correction did not hold'
assert _res.max() <= SW_OUT_MAX, f'(!) a half-hourly mean exceeds SW_OUT_MAX ({SW_OUT_MAX})'
print(f'{FIELD} at {RESAMPLING_FREQ}: {_res.count():,} of {len(_res):,} half hours have a value '
      f'({_res.count() / len(_res):.2%}), range {_res.min():.2f} to {_res.max():.2f} W m-2')
display(_res.groupby(_res.index.year).agg(['count', 'mean', 'max']).round(2))

## ⬆️ Upload data to database

**Re-uploading overwrites the same variant — safe to re-run.** With `delete_from_db_before_upload=True` (below), the upload first *deletes*, then writes. The delete is scoped to the exact match `_measurement` + `varname` + `data_version` (`meteoscreening_diive`) over the uploaded time range, so re-screening a period replaces only its previous screened result. It never touches the raw data (different `data_version`, and a different `_raw` bucket), other variables, or other data versions. The delete-first step (rather than a plain overwrite) matters because InfluxDB keys a point by its full tag set: if a tag changed between runs (e.g. `units`, `gain`, `offset`), a plain write would leave the old point as a **duplicate** — the delete removes it regardless of tags.

In [ ]:
print(f'Uploading to bucket {BUCKET_PROCESSED}')
for v in mscr.resampled_detailed.keys():
    dbc.upload_singlevar(
        to_bucket=BUCKET_PROCESSED,
        to_measurement=assigned_measurements[v],
        var_df=mscr.resampled_detailed[v],
        timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
        delete_from_db_before_upload=True,
    )

### Verify upload
Download the just-uploaded data back and confirm the time resolution and timestamps. The offset is the same `TIMEZONE_OFFSET_TO_UTC_HOURS`, so the timestamps below should again be local `TIMESTAMP_END` — matching what you screened.

In [ ]:
# Fresh variable names so the screened originals (data_detailed etc.) are not overwritten:
check_simple, check_detailed, check_measurements = dbc.download(
    bucket=BUCKET_PROCESSED,
    measurements=[MEASUREMENT],
    fields=FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version='meteoscreening_diive',
)
check_simple

In [ ]:
for v in check_detailed.keys():
    idx = check_detailed[v].index
    freq = dv.times.DetectFrequency(index=idx, verbose=True).get()
    status = 'PASSED' if freq == RESAMPLING_FREQ else '(!) FAILED'
    print(f'{status} - {v}: freq={freq}, first={idx[0]}, last={idx[-1]}')

# The round trip must return what was uploaded. A mismatch here means the upload wrote a
# different time range or a duplicate point survived the delete.
_back = check_detailed[FIELD][FIELD]
_uploaded = mscr.resampled_detailed[FIELD][FIELD]
_common = _back.index.intersection(_uploaded.index)
assert len(_common) > 0, '(!) nothing came back from the processed bucket'
assert not _back.index.has_duplicates, '(!) duplicate timestamps in the processed bucket'
_diff = (_back.loc[_common] - _uploaded.loc[_common]).abs().max()
print(f'Round trip: {len(_common):,} shared half hours, largest difference {_diff:.6f} W m-2')

## ✅ End of notebook

In [ ]:
_end = datetime.now()
print(f"Finished: {_end.strftime('%Y-%m-%d %H:%M:%S')}  "
      f"(runtime {str(_end - NOTEBOOK_START).split('.')[0]})")

***
### 📝 Notes for this variable

- **Mean, not sum.** `RESAMPLING_AGG = 'mean'` — a radiative flux density is a state at the half hour, not an accumulation.
- **Units are W m^-2^.** The unit tag read from the database is printed by the coverage report; check it against the magnitude of the data rather than trusting either alone.
- **Sensor identity.** The downward-facing pyranometer of the four-component radiometer at 47 m: Kipp &amp; Zonen **CNR4**, SN 212965, `SW_OUT` sensitivity 14.38 µV W^-1^ m^2^, installed 14 December 2021; a **CNR1**, SN 020484, was in place before that. Whether the CNR1's outgoing channel was ever logged is answered by the coverage report — see below.
- **Coverage is settled at 2020-2025.** `SW_OUT_T1_47_1` and `LW_OUT_T1_47_1` both begin 2020-01-02 00:52 and nothing precedes either, so the CNR1 outgoing channels were never ingested (`PLAN.md` §3.2). `SW_OUT`, `ALB` and a four-component `NETRAD` are therefore all 2020-2025 products. The *Coverage and sensor eras* section asserts that start date on every run, so a re-ingestion that moved it stops the notebook instead of quietly changing the product period.
- **The 24-day window 14 Dec 2021 - 7 Jan 2022 is unverified and is not removed here.** The CNR4 was installed on the first date, the logger received its four sensitivities on the second. If the new instrument was read through the CNR1 multiplier, those values are wrong by the ratio of the sensitivities — smooth, in-range and invisible to every test in this notebook. It is a calibration question, it is decided on the half-hourly series against the other three components, and the count of affected records is printed above. See `docs/Instrumentation.md`.
- **A negative nighttime reading is not an error.** It is the pyranometer's own thermal offset, it is the input the zero-offset correction is estimated from, and any screening test that removes it also biases the correction. That is why the statistical tests are off and why `SW_OUT_MIN` is well below zero rather than at it.
- **The zero-offset correction fills gaps, and this notebook undoes that.** `correction_remove_nighttime_zero_offset()` sets the whole nighttime index to zero, missing records included. The mask is captured before the correction and re-applied after it, so the uploaded series carries a value only where one was measured. Zero-filling the nights is a reasonable product decision — it belongs in `30_PRODUCTS`, where it can be flagged, not here where it would be indistinguishable from measurement.
- **Deferred to `30_PRODUCTS/`, on the 30MIN data:** the albedo cross-check against `SW_IN_T1_47_1` (the sharpest available test of this channel, since `SW_OUT / SW_IN` is bounded, seasonal and physically interpretable in a way neither series is alone), the CNR1 → CNR4 continuity question, the 24-day window, and any gap handling. This notebook exports gaps and does not fill them.

### ♻️ Sibling notebooks
`SW_IN/SW_IN_T1_47_1_*.ipynb` screen the **incoming** channel of the same instrument, and `LW_IN/LW_IN_T1_47_1_*.ipynb` the incoming longwave. They are the right notebooks to read for how this site's radiation record behaves — and the wrong ones to copy removal windows from: the four channels of a CNR are four detectors, and a fault in one is not evidence about the others.

`SW_OUT_BC_M1_2_1`, the subcanopy outgoing channel, is a different measurement level and is listed in `PLAN.md` §3.3 as undecided. It is not screened by this notebook and must not be merged with this series.